##MOVIE RECOMMENDATION SYSTEM
#Content-Based Filtering

In [3]:
import pandas as pd
import numpy as np

from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [4]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv", nrows=500000)

#Preview Datasets

In [5]:
print("Movies dataset:")
display(movies.head())

print("Ratings dataset:")
display(ratings.head())

print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)

Movies dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings dataset:


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


Movies shape: (62423, 3)
Ratings shape: (500000, 4)


In [6]:
movies["genres"] = movies["genres"].fillna("")

print("Missing values in movies:")
print(movies.isnull().sum())

print("\nMissing values in ratings:")
print(ratings.isnull().sum())

Missing values in movies:
movieId    0
title      0
genres     0
dtype: int64

Missing values in ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


#Create TF-IDF Matrix Model

In [7]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["genres"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (62423, 23)


#Ratings Summary

In [8]:
ratings_summary = ratings.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    num_ratings=("rating", "count")
).reset_index()

display(ratings_summary.head())

,movieId,avg_rating,num_ratings
0,1,3.901028,1167
1,2,3.312632,475
2,3,3.144128,281
3,4,2.932432,37
4,5,3.156627,249


#Merge Movie & Rating Data

In [9]:
movies = movies.merge(ratings_summary, on="movieId", how="left")
movies["avg_rating"] = movies["avg_rating"].fillna(0)
movies["num_ratings"] = movies["num_ratings"].fillna(0)

display(movies.head())

,movieId,title,genres,avg_rating,num_ratings
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.901028,1167.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.312632,475.0
2,3,Grumpier Old Men (1995),Comedy|Romance,3.144128,281.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.932432,37.0
4,5,Father of the Bride Part II (1995),Comedy,3.156627,249.0


In [10]:
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()
print("Movie index mapping created.")

Movie index mapping created.


#Recommendation Function

In [11]:
def recommend_movies(title, top_n=10):
    # Normalize user input
    user_input = title.strip().lower()

    # Create lowercase title mapping
    lower_title_map = {movie.lower(): movie for movie in indices.index}

    # 1. Exact match ignoring case
    if user_input in lower_title_map:
        chosen_title = lower_title_map[user_input]

    else:
        # 2. Partial match ignoring case
        matches = [movie for movie in indices.index if user_input in movie.lower()]

        if len(matches) == 0:
            return f"Movie '{title}' not found in dataset."

        # If multiple matches, use the first one
        chosen_title = matches[0]
        print("Movie not found exactly. Using closest match:")
        print(chosen_title)

    idx = indices[chosen_title]

    # Compute similarity only for the selected movie
    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()

    # Get top similar movies
    sim_indices = sim_scores.argsort()[::-1][1:21]

    recommendations = movies.iloc[sim_indices][["title", "genres", "avg_rating", "num_ratings"]].copy()

    recommendations = recommendations.sort_values(
        by=["avg_rating", "num_ratings"],
        ascending=False
    )

    print(f"\nRecommendations based on: {chosen_title}")
    return recommendations.head(top_n)

#Recommender Test

In [12]:
movie_name = "Toy Story (1995)"
print(f"Recommendations for: {movie_name}")
display(recommend_movies(movie_name))

Recommendations for: Toy Story (1995)

Recommendations based on: Toy Story (1995)


,title,genres,avg_rating,num_ratings
52826,Tangled: Before Ever After (2017),Adventure|Animation|Children|Comedy|Fantasy,4.000000,1.0
22633,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,4.000000,1.0
4780,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,3.872794,680.0
43614,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,3.821839,87.0
3912,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,3.619760,167.0
17431,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,3.166667,3.0
12969,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,3.062500,8.0
9949,DuckTales: The Movie - Treasure of the Lost La...,Adventure|Animation|Children|Comedy|Fantasy,3.000000,3.0
11604,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,2.903409,88.0
10773,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,2.875000,8.0


#Interactive Movie Recommendation

In [15]:
while True:
    movie_name = input("Enter a movie title (or type 'exit' to stop): ")

    if movie_name.lower() == "exit":
        print("Exiting recommendation system.")
        break

    result = recommend_movies(movie_name)

    if isinstance(result, str):
        print(result)
    else:
        display(result)

Enter a movie title (or type 'exit' to stop): Jumanji (1995)

Recommendations based on: Jumanji (1995)


,title,genres,avg_rating,num_ratings
12042,"Water Horse: Legend of the Deep, The (2007)",Adventure|Children|Fantasy,3.350000,10.0
15610,Chronicles of Narnia: The Voyage of the Dawn T...,Adventure|Children|Fantasy,3.345238,42.0
15298,Alice in Wonderland (1933),Adventure|Children|Fantasy,3.166667,3.0
12347,"Chronicles of Narnia: Prince Caspian, The (2008)",Adventure|Children|Fantasy,3.159420,69.0
20088,Percy Jackson: Sea of Monsters (2013),Adventure|Children|Fantasy,2.944444,18.0
23835,Seventh Son (2014),Adventure|Children|Fantasy,2.750000,8.0
15765,"Polar Bear King, The (Kvitebjørn Kong Valemon)...",Adventure|Children|Fantasy,1.000000,1.0
55115,Prince Caspian and the Voyage of the Dawn Trea...,Adventure|Children|Fantasy,0.000000,0.0
26483,The Cave of the Golden Rose (1991),Adventure|Children|Fantasy,0.000000,0.0
26245,Le petit poucet (2001),Adventure|Children|Fantasy,0.000000,0.0


Enter a movie title (or type 'exit' to stop): exit
Exiting recommendation system.
